In [ ]:
# PRECURSOR EXTRACTION

In [ ]:
# ============================================
# PRECURSOR EXTRACTION SCRIPT
# ============================================

import pandas as pd
from pathlib import Path

# ============================================
# PARAMETERS
# ============================================

# Target precursor m/z values
TARGET_MZ = [
    313.13, 327.11, 329.12, ........
]

# Mass tolerance (± Da)
TOLERANCE = 0.25


# ============================================
# FUNCTION: EXTRACT PRECURSORS
# ============================================

def extract_precursors(csv_path, out_dir):
    """
    Extract rows whose 'Measured Mass' falls within ±TOLERANCE
    of any value in TARGET_MZ.

    Parameters:
    ----------
    csv_path : str or Path
        Path to input CSV file

    out_dir : str or Path
        Directory to save output file

    Returns:
    -------
    out_file : Path
        Output CSV path

    n_rows : int
        Number of extracted rows
    """

    csv_path = Path(csv_path)
    out_dir = Path(out_dir)

    # Create output folder if it doesn't exist
    out_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------
    # Load dataset
    # -------------------------
    df = pd.read_csv(csv_path)

    # Check required column
    if "Measured Mass" not in df.columns:
        raise ValueError(f"'Measured Mass' column not found in {csv_path.name}")

    matched_rows = []

    # -------------------------
    # Loop through target masses
    # -------------------------
    for mz in TARGET_MZ:

        # Filter rows within ± tolerance
        mask = (
            (df["Measured Mass"] >= mz - TOLERANCE) &
            (df["Measured Mass"] <= mz + TOLERANCE)
        )

        subset = df[mask].copy()

        # If matches found, store them
        if not subset.empty:
            subset["Target_mz"] = mz  # track matched precursor
            matched_rows.append(subset)

        print(f"Target {mz:.2f} → {len(subset)} matches")

    # -------------------------
    # Combine results
    # -------------------------
    if matched_rows:
        precursor_df = pd.concat(matched_rows, ignore_index=True)

        # Remove duplicate rows (important!)
        precursor_df = precursor_df.drop_duplicates()
    else:
        precursor_df = pd.DataFrame(columns=df.columns)

    # -------------------------
    # Save output
    # -------------------------
    out_file = out_dir / f"{csv_path.stem}_precursors.csv"
    precursor_df.to_csv(out_file, index=False)

    return out_file, len(precursor_df)


# ============================================
# MAIN EXECUTION
# ============================================

if __name__ == "__main__":

    # -------------------------
    # INPUT DATASETS
    # -------------------------
    datasets = [
        r"...Open ocean/Station_14_10m.csv",
        # Add more files here if needed
        # r".../Station_14_10m.csv",
        # r".../Station_2_1000m.csv",
    ]

    # -------------------------
    # OUTPUT DIRECTORY
    # -------------------------
    output_folder = r".../Open ocean/Precursor_data"

    # -------------------------
    # RUN EXTRACTION
    # -------------------------
    print("\n🚀 Starting precursor extraction...\n")

    for file in datasets:
        try:
            out_file, nrows = extract_precursors(file, output_folder)

            print(f"\n✔ File processed: {file}")
            print(f"   Extracted rows: {nrows}")
            print(f"   Output saved to: {out_file}\n")

        except Exception as e:
            print(f"\n❌ Error processing {file}: {e}\n")

    print("✅ DONE\n")

In [ ]:
# CHO FILTERING

In [ ]:
# ============================================
# CHO FILTERING SCRIPT
# ============================================

import pandas as pd
from pathlib import Path

# ============================================
# FUNCTION: KEEP CHO ONLY
# ============================================

def keep_CHO_only(csv_path, out_dir):
    """
    Filters a precursor dataset to keep only CHO compounds.

    Parameters:
    ----------
    csv_path : str or Path
        Path to input precursor CSV file

    out_dir : str or Path
        Output directory

    Returns:
    -------
    out_file : Path
        Saved filtered file

    n_rows : int
        Number of CHO rows
    """

    csv_path = Path(csv_path)
    out_dir = Path(out_dir)

    # Create output folder if needed
    out_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------
    # Load dataset
    # -------------------------
    df = pd.read_csv(csv_path)

    # Check required column
    if "Group" not in df.columns:
        raise ValueError(f"'Group' column not found in {csv_path.name}")

    # -------------------------
    # Filter CHO group
    # -------------------------
    df_CHO = df[df["Group"] == "CHO"].copy()

    # -------------------------
    # Save output
    # -------------------------
    out_file = out_dir / f"{csv_path.stem}_CHO.csv"
    df_CHO.to_csv(out_file, index=False)

    return out_file, len(df_CHO)


# ============================================
# MAIN EXECUTION
# ============================================

if __name__ == "__main__":

    # -------------------------
    # INPUT FILES
    # -------------------------
    precursor_files = [
        r".../Open ocean/Station 14_4000m_precursors.csv",
        
    ]

    # -------------------------
    # OUTPUT DIRECTORY
    # -------------------------
    output_folder = r".../Open ocean/Precursor_data/CHO_only"

    # -------------------------
    # RUN FILTER
    # -------------------------
    print("\n🚀 Starting CHO filtering...\n")

    for file in precursor_files:
        try:
            out_file, nrows = keep_CHO_only(file, output_folder)

            print(f"✔ File processed: {Path(file).name}")
            print(f"   CHO rows kept: {nrows}")
            print(f"   Output: {out_file}\n")

        except Exception as e:
            print(f"❌ Error with {file}: {e}\n")

    print("✅ DONE\n")

In [ ]:
# MS2 FILTERING

In [ ]:
# ============================================
# MS2 FILTERING SCRIPT
# ============================================

import pandas as pd
from pathlib import Path
import re

# ============================================
# PARAMETERS
# ============================================

PRECURSOR_MASSES = [
    313.13, 327.11, 329.12, .....
]

TOLERANCE_DA = 0.25
SN_THRESHOLD = 10

INPUT_FOLDER = r"C:/...../Open ocean/St 14_10m/Raw csv St 14_10m"
OUTPUT_FOLDER = r"C:/..../Open ocean/St 14_10m/filtered St 14_10m"


# ============================================
# HELPER FUNCTIONS
# ============================================

def extract_integer_mass(filename):
    """
    Extract integer mass from filename.
    Example:
        'Station_14_10m_313' → 313
    """
    match = re.search(r'_(\d+)$', filename)
    return int(match.group(1)) if match else None


def find_closest_precursor(int_mass):
    """
    Match integer mass to closest known precursor mass.
    """
    return min(PRECURSOR_MASSES, key=lambda x: abs(x - int_mass))


# ============================================
# MAIN FILTER FUNCTION
# ============================================

def filter_ms2_dataset(csv_path, out_dir):

    csv_path = Path(csv_path)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------
    # 1. Identify precursor
    # -------------------------
    int_mass = extract_integer_mass(csv_path.stem)

    if int_mass is None:
        raise ValueError("No integer mass found in filename")

    precursor_mass = find_closest_precursor(int_mass)

    # -------------------------
    # 2. Load data
    # -------------------------
    df = pd.read_csv(csv_path)

    # Check required columns
    for col in ["m/z", "S/N"]:
        if col not in df.columns:
            raise ValueError(f"Missing column '{col}' in {csv_path.name}")

    # -------------------------
    # 3. Remove fragments above precursor
    # -------------------------
    df = df[df["m/z"] <= precursor_mass + TOLERANCE_DA]

    # -------------------------
    # 4. Remove low S/N fragments
    # ONLY below precursor window
    # -------------------------
    mask_remove = (
        (df["m/z"] < precursor_mass - TOLERANCE_DA) &
        (df["S/N"] < SN_THRESHOLD)
    )

    df_filtered = df.loc[~mask_remove].copy()

    # -------------------------
    # 5. Add precursor info
    # -------------------------
    df_filtered["Precursor_mz"] = precursor_mass

    # -------------------------
    # 6. Save output
    # -------------------------
    out_file = out_dir / f"{csv_path.stem}_filtered.csv"
    df_filtered.to_csv(out_file, index=False)

    return out_file, len(df_filtered), precursor_mass


# ============================================
# RUN PIPELINE
# ============================================

if __name__ == "__main__":

    print("\n🚀 Starting MS2 filtering...\n")

    for csv_file in Path(INPUT_FOLDER).glob("*.csv"):
        try:
            out_file, nrows, pmz = filter_ms2_dataset(csv_file, OUTPUT_FOLDER)

            print(f"✔ {csv_file.name}")
            print(f"   Precursor: {pmz:.2f}")
            print(f"   Rows kept: {nrows}")
            print(f"   Output: {out_file}\n")

        except Exception as e:
            print(f"⚠ Skipped {csv_file.name}: {e}\n")

    print("✅ DONE\n")

In [ ]:
# BFS MASS NETWORK USING NEUTRAL LOSSES

In [ ]:
# ============================================
# BFS MASS NETWORK USING NEUTRAL LOSSES
# ============================================

import pandas as pd
from collections import deque


# ============================================
# FUNCTION: BUILD NETWORK FROM START MASS
# ============================================

def process_starting_mass(start_mass, neutral_losses, dataset, tolerance=0.001):
    """
    Build a mass network starting from a precursor using neutral losses.

    Parameters:
    ----------
    start_mass : float
        Starting precursor m/z

    neutral_losses : DataFrame
        Must contain column 'mf' (mass of neutral loss)
        Optional: 'Formula', 'Name'

    dataset : DataFrame
        Must contain column 'm/z' (fragment masses)

    tolerance : float
        Matching tolerance (Da)

    Returns:
    -------
    DataFrame of edges:
        Parent_Mass → Child_Mass via Neutral Loss
    """

    queue = deque([start_mass])
    visited = set()
    matched_entries = []

    # Convert to arrays (faster)
    ds_masses = dataset['m/z'].values
    loss_masses = neutral_losses['mf'].values

    has_formula = 'Formula' in neutral_losses.columns
    has_name = 'Name' in neutral_losses.columns

    # ============================================
    # BFS TRAVERSAL
    # ============================================

    while queue:
        current_mass = queue.popleft()

        key = round(float(current_mass), 6)
        if key in visited:
            continue
        visited.add(key)

        # Only allow fragmentation (child ≤ parent)
        candidates = ds_masses[ds_masses <= current_mass + tolerance]

        for child_mass in candidates:

            mass_diff = current_mass - child_mass

            # Compare with all neutral losses
            for i, loss_mass in enumerate(loss_masses):

                if abs(mass_diff - loss_mass) <= tolerance:

                    row = {
                        "Parent_Mass": float(current_mass),
                        "Child_Mass": float(child_mass),
                        "Matched_Loss_Mass": float(loss_mass)
                    }

                    if has_formula:
                        row["Neutral_Loss_Formula"] = neutral_losses.iloc[i]["Formula"]

                    if has_name:
                        row["Neutral_Loss_Name"] = neutral_losses.iloc[i]["Name"]

                    matched_entries.append(row)

                    # Continue traversal from child
                    queue.append(child_mass)

    return pd.DataFrame(matched_entries)


# ============================================
# MAIN SCRIPT
# ============================================

def main():

    # -------------------------
    # INPUT FILES
    # -------------------------

    starting_masses = pd.read_csv("313.csv")   # must contain: m/z
    neutral_losses = pd.read_csv("PMD_CHO.csv")  # must contain: mf
    dataset = pd.read_csv(
        r"C:..../Open ocean/Station2_1000m_313_filtered.csv"
    )

    # -------------------------
    # VALIDATION
    # -------------------------

    for col, df in [
        ("m/z", starting_masses),
        ("m/z", dataset),
        ("mf", neutral_losses)
    ]:
        if col not in df.columns:
            raise ValueError(f"Missing required column '{col}'")

    # Convert to numeric
    starting_masses["m/z"] = pd.to_numeric(starting_masses["m/z"], errors="coerce")
    dataset["m/z"] = pd.to_numeric(dataset["m/z"], errors="coerce")
    neutral_losses["mf"] = pd.to_numeric(neutral_losses["mf"], errors="coerce")

    # Drop NaNs
    starting_masses = starting_masses.dropna(subset=["m/z"])
    dataset = dataset.dropna(subset=["m/z"])
    neutral_losses = neutral_losses.dropna(subset=["mf"])

    # -------------------------
    # PARAMETERS
    # -------------------------

    tolerance = 0.001

    # -------------------------
    # RUN BFS
    # -------------------------

    print("\n🚀 Building mass networks...\n")

    for _, row in starting_masses.iterrows():

        start_mass = float(row["m/z"])
        print(f"Processing: {start_mass:.5f}")

        result_df = process_starting_mass(
            start_mass,
            neutral_losses,
            dataset,
            tolerance
        )

        if not result_df.empty:
            out_file = f"connected_masses_bfs_{start_mass:.5f}.csv"

            result_df.sort_values(
                ["Parent_Mass", "Child_Mass", "Matched_Loss_Mass"]
            ).to_csv(out_file, index=False)

            print(f"✔ Saved: {out_file} ({len(result_df)} edges)\n")
        else:
            print("⚠ No connections found\n")


# ============================================
# ENTRY POINT
# ============================================

if __name__ == "__main__":
    main()

In [ ]:
# EXTRACT UNIQUE MASSES FROM BFS NETWORK

In [ ]:
# ============================================
# EXTRACT UNIQUE MASSES FROM BFS NETWORK FILES
# ============================================

import os
import pandas as pd

# ============================================
# PARAMETERS
# ============================================

folder_path = r"C:/...../Open ocean"


# ============================================
# MAIN PROCESS
# ============================================

print("\n🚀 Extracting unique masses from BFS outputs...\n")

for filename in os.listdir(folder_path):

    # Only process BFS output files
    if filename.startswith("connected_masses_bfs_") and filename.endswith(".csv"):

        file_path = os.path.join(folder_path, filename)

        try:
            df = pd.read_csv(file_path)

            # -------------------------
            # Check required columns
            # -------------------------
            if not {"Parent_Mass", "Child_Mass"}.issubset(df.columns):
                print(f"⚠ Skipping {filename}: missing required columns")
                continue

            # -------------------------
            # Extract unique masses
            # -------------------------
            unique_masses = pd.unique(
                df[["Parent_Mass", "Child_Mass"]].values.ravel("K")
            )

            # Convert to DataFrame
            masses_df = pd.DataFrame(unique_masses, columns=["Unique_Mass"])

            # Remove NaN (just in case)
            masses_df = masses_df.dropna()

            # Sort values
            masses_df = masses_df.sort_values("Unique_Mass").reset_index(drop=True)

            # -------------------------
            # Save output
            # -------------------------
            out_name = filename.replace(
                "connected_masses_bfs_", "unique_masses_"
            )
            out_path = os.path.join(folder_path, out_name)

            masses_df.to_csv(out_path, index=False)

            print(f"✔ {filename}")
            print(f"   Unique masses: {len(masses_df)}")
            print(f"   Saved → {out_name}\n")

        except Exception as e:
            print(f"❌ Error with {filename}: {e}\n")

print("✅ DONE\n")

In [ ]:
# MASS DIFFERENCE → TRANSFORMATION MATCHING

In [ ]:
# ============================================
# MASS DIFFERENCE → TRANSFORMATION MATCHING
# ============================================

import os
import pandas as pd
from itertools import combinations

# ============================================
# PATHS
# ============================================

unique_folder = r"C:/Users/kusha/OneDrive/Desktop/FTMS data analysis/Open ocean"
key_path = r"C:/Users/kusha/OneDrive/Desktop/FTMS data analysis/Open ocean/PMD_CHO.csv"
output_path = unique_folder

# ============================================
# PARAMETERS
# ============================================

tol = 0.001
min_diff, max_diff = 1, 766
round_diff_decimals = 6


# ============================================
# HELPERS
# ============================================

def read_mz_column(df):
    """Find and return valid mass column."""
    candidates = ["Unique_Mass", "m/z", "mz", "Mass"]
    for c in candidates:
        if c in df.columns:
            s = pd.to_numeric(df[c], errors="coerce")
            return s[(s > 0) & s.notna()]
    raise ValueError(f"No valid mass column found: {candidates}")


# ============================================
# MAIN TRANSFORMATION FUNCTION
# ============================================

def calculate_transformations_single_sample(mz_series, keys, out_dir, sample_id):

    mz_list = sorted(set(float(x) for x in mz_series))
    print(f"\n🔬 Processing {sample_id}")
    print(f"   Total masses: {len(mz_list)}")

    # Extract valid neutral losses
    key_pairs = list(zip(keys["Formula"], pd.to_numeric(keys["mf"], errors="coerce")))
    key_pairs = [(f, m) for f, m in key_pairs if pd.notna(m)]

    if not key_pairs:
        raise ValueError("No valid neutral loss masses found")

    result_rows = []

    # ============================================
    # ALL-VS-ALL MASS DIFFERENCES
    # ============================================

    for x, y in combinations(mz_list, 2):

        diff = abs(x - y)

        # Skip irrelevant differences
        if not (min_diff <= diff <= max_diff):
            continue

        # Match to neutral losses
        for formula, mf in key_pairs:

            if abs(diff - mf) <= tol:

                result_rows.append(
                    (x, y, round(diff, round_diff_decimals), formula, mf)
                )

    if not result_rows:
        print("⚠ No transformations found")
        return

    # ============================================
    # SAVE MATCHES
    # ============================================

    result_df = pd.DataFrame(
        result_rows,
        columns=["Feature_X", "Feature_Y", "Difference", "Formula", "mf"]
    )

    result_df["SampleID"] = sample_id

    out_file = os.path.join(out_dir, f"transformations_{sample_id}.csv")
    result_df.to_csv(out_file, index=False)

    print(f"✔ Saved transformations ({len(result_df)} rows)")

    # ============================================
    # COUNT TRANSFORMATIONS
    # ============================================

    counts = (
        result_df.groupby("Formula")
        .size()
        .reset_index(name="Counts")
        .sort_values("Counts", ascending=False)
        .reset_index(drop=True)
    )

    total = counts["Counts"].sum()
    counts["Perc_Counts"] = counts["Counts"] / total

    out_counts = os.path.join(out_dir, f"counts_{sample_id}.csv")
    counts.to_csv(out_counts, index=False)

    print(f"✔ Saved counts\n")


# ============================================
# LOAD KEY FILE
# ============================================

keys = pd.read_csv(key_path)

if "Formula" not in keys.columns or "mf" not in keys.columns:
    raise ValueError("Key file must contain 'Formula' and 'mf'")


# ============================================
# RUN FOR ALL FILES
# ============================================

print("\n🚀 Starting transformation analysis...\n")

for fname in os.listdir(unique_folder):

    if fname.startswith("unique_masses_") and fname.endswith(".csv"):

        fpath = os.path.join(unique_folder, fname)

        try:
            df = pd.read_csv(fpath)

            mz_series = read_mz_column(df)
            sample_id = os.path.splitext(fname)[0]

            calculate_transformations_single_sample(
                mz_series, keys, output_path, sample_id
            )

        except Exception as e:
            print(f"⚠ Skipped {fname}: {e}")

print("✅ DONE\n")

In [ ]:
# REARRANGE TRANSFORMATION EDGES

In [ ]:
# ============================================
# REARRANGE TRANSFORMATION EDGES (UNDIRECTED)
# ============================================

import os
import pandas as pd
from glob import glob

# ============================================
# PATHS
# ============================================

input_dir = r"C:/Users/kusha/OneDrive/Desktop/FTMS data analysis/Open ocean"
output_dir = os.path.join(input_dir, "rearranged")
os.makedirs(output_dir, exist_ok=True)

# ============================================
# PARAMETERS
# ============================================

ROUND_DECIMALS = 6

# ============================================
# FIND FILES
# ============================================

files = sorted(glob(os.path.join(input_dir, "transformations_*.csv")))
print(f"\n🔍 Found {len(files)} transformation files\n")

# ============================================
# PROCESS FILES
# ============================================

for file in files:
    try:
        df = pd.read_csv(file)

        print(f"⚙ Processing: {os.path.basename(file)}")

        # ------------------------------------
        # Detect column names
        # ------------------------------------
        if {"Feature_X", "Feature_Y"}.issubset(df.columns):
            x_col, y_col = "Feature_X", "Feature_Y"
        elif {"Source", "Target"}.issubset(df.columns):
            x_col, y_col = "Source", "Target"
        else:
            print(f"⚠ Skipping: missing required columns\n")
            continue

        # ------------------------------------
        # Convert to numeric
        # ------------------------------------
        df[x_col] = pd.to_numeric(df[x_col], errors="coerce")
        df[y_col] = pd.to_numeric(df[y_col], errors="coerce")

        df = df.dropna(subset=[x_col, y_col]).copy()

        # ------------------------------------
        # Enforce ordering: X ≥ Y
        # ------------------------------------
        swap_mask = df[x_col] < df[y_col]

        df.loc[swap_mask, [x_col, y_col]] = (
            df.loc[swap_mask, [y_col, x_col]].values
        )

        # ------------------------------------
        # Create unique edge key
        # ------------------------------------
        df["_edge_key"] = list(
            zip(
                df[x_col].round(ROUND_DECIMALS),
                df[y_col].round(ROUND_DECIMALS)
            )
        )

        # ------------------------------------
        # Remove duplicate edges
        # ------------------------------------
        df = df.drop_duplicates(subset=["_edge_key"]).copy()
        df = df.drop(columns=["_edge_key"])

        # ------------------------------------
        # (OPTIONAL) sort edges
        # ------------------------------------
        df = df.sort_values([x_col, y_col]).reset_index(drop=True)

        # ------------------------------------
        # Save output
        # ------------------------------------
        out_name = f"rearranged_{os.path.basename(file)}"
        out_path = os.path.join(output_dir, out_name)

        df.to_csv(out_path, index=False)

        print(f"✔ Saved: {out_name}")
        print(f"   Rows: {len(df)}\n")

    except Exception as e:
        print(f"❌ Failed: {os.path.basename(file)} → {e}\n")

print("✅ DONE\n")

In [ ]:
# MAX CO2 PATHWAY EXTRACTION (SEQUENTIAL)

In [ ]:
# ============================================
# MAX CO2 PATHWAY EXTRACTION (FINAL STEP)
# ============================================

import os
import re
import pandas as pd
from collections import deque

# ============================================
# PATHS
# ============================================

input_folder  = r"C:/Users/kusha/OneDrive/Desktop/FTMS data analysis/Open ocean/rearranged"
output_folder = r"C:/Users/kusha/OneDrive/Desktop/FTMS data analysis/Open ocean/CO2_max_paths"
os.makedirs(output_folder, exist_ok=True)

ROUND_DECIMALS = 5

# ============================================
# FORMULA HANDLING
# ============================================

def norm_formula(s):
    """Normalize formula labels."""
    return str(s).strip().upper().replace(" ", "").replace("₂", "2")

def is_co2_like(label):
    """Detect CO2 and isotopic variants."""
    n = norm_formula(label)
    return n == "CO2" or n.startswith("CO2(")

# ============================================
# HELPERS
# ============================================

def extract_starting_mass(filename):
    """Extract starting mass from filename."""
    match = re.search(r"_(\d+\.\d+)", filename)
    return float(match.group(1)) if match else None


def load_graph(df):
    """Convert dataframe to adjacency graph."""
    df["Feature_X"] = pd.to_numeric(df["Feature_X"], errors="coerce")
    df["Feature_Y"] = pd.to_numeric(df["Feature_Y"], errors="coerce")
    df = df.dropna(subset=["Feature_X", "Feature_Y"])

    df["Feature_X"] = df["Feature_X"].round(ROUND_DECIMALS)
    df["Feature_Y"] = df["Feature_Y"].round(ROUND_DECIMALS)

    if "Formula" not in df.columns:
        df["Formula"] = ""

    graph = {}

    for _, row in df.iterrows():
        src = row["Feature_X"]
        tgt = row["Feature_Y"]
        formula = str(row["Formula"]).strip()

        graph.setdefault(src, []).append((tgt, formula))

    return graph

# ============================================
# BFS FOR MAX CO2 PATH
# ============================================

def find_max_co2_paths(graph, start_mass):

    start_mass = round(start_mass, ROUND_DECIMALS)

    queue = deque([(start_mass, [start_mass], [], 0)])

    max_co2 = 0
    best_paths = []

    visited_states = set()

    while queue:
        current_mass, path, formulas, co2_count = queue.popleft()

        state = (current_mass, co2_count)
        if state in visited_states:
            continue
        visited_states.add(state)

        # Update best result
        if co2_count > max_co2:
            max_co2 = co2_count
            best_paths = [(path, formulas, co2_count)]
        elif co2_count == max_co2:
            best_paths.append((path, formulas, co2_count))

        # Explore neighbors
        for neighbor, formula in graph.get(current_mass, []):

            new_co2 = co2_count + (1 if is_co2_like(formula) else 0)

            queue.append(
                (neighbor, path + [neighbor], formulas + [formula], new_co2)
            )

    return best_paths


# ============================================
# MAIN LOOP
# ============================================

print("\n🚀 Extracting MAX CO2 pathways...\n")

for file in sorted(os.listdir(input_folder)):

    if not file.endswith(".csv"):
        continue

    start_mass = extract_starting_mass(file)

    if start_mass is None:
        print(f"⚠ Skipping {file} (no start mass)")
        continue

    fpath = os.path.join(input_folder, file)

    try:
        df = pd.read_csv(fpath)

        if not {"Feature_X", "Feature_Y"}.issubset(df.columns):
            print(f"⚠ Skipping {file}: missing columns")
            continue

        graph = load_graph(df)

        best_paths = find_max_co2_paths(graph, start_mass)

        if not best_paths:
            print(f"⚠ No paths found: {file}")
            continue

        # Save results
        results = []

        for path, formulas, co2_count in best_paths:
            results.append({
                "File": file,
                "Start_Mass": start_mass,
                "CO2_Count": co2_count,
                "Mass_Path": " -> ".join(f"{m:.5f}" for m in path),
                "Formula_Path": " -> ".join(formulas)
            })

        out_df = pd.DataFrame(results)

        out_name = file.replace(".csv", "_max_CO2_path.csv")
        out_path = os.path.join(output_folder, out_name)

        out_df.to_csv(out_path, index=False)

        print(f"✔ {file} → max CO2 = {results[0]['CO2_Count']}")

    except Exception as e:
        print(f"❌ Error in {file}: {e}")

print("\n✅ DONE\n")

In [ ]:
# CO2 VARIANT PATHWAY EXTRACTION (NON-SEQUENTIAL)

In [ ]:
# ============================================
# CO2 VARIANT PATHWAY EXTRACTION (ALL PATHS)
# ============================================

import os
import re
import pandas as pd

ROUND_DECIMALS = 5

# ============================================
# FORMULA HANDLING
# ============================================

def norm_formula(s: str) -> str:
    return str(s).strip().upper().replace(" ", "").replace("₂", "2")


def is_co2_variant(s: str) -> bool:
    """Accept CO2 and isotopic variants like CO2(18,18)."""
    n = norm_formula(s)
    return n == "CO2" or n.startswith("CO2(")


# ============================================
# RECURSIVE TRAVERSAL
# ============================================

def trace_co2_variants(df_co2, current_mass, pathway, step, visited):

    if current_mass in visited:
        return []

    visited.add(current_mass)

    sub_df = df_co2[df_co2["Feature_X_round"] == current_mass]

    # Terminal node
    if sub_df.empty:
        pathway[f"Step_{step}_Mass"] = current_mass
        return [pathway]

    paths = []

    for _, row in sub_df.iterrows():

        new_path = pathway.copy()

        new_path[f"Step_{step}_Mass"] = row["Feature_X_round"]
        new_path[f"Step_{step}_Loss"] = row["Formula"]

        next_mass = row["Feature_Y_round"]

        paths.extend(
            trace_co2_variants(
                df_co2,
                next_mass,
                new_path,
                step + 1,
                visited.copy()
            )
        )

    return paths


# ============================================
# MAIN ANALYSIS FUNCTION
# ============================================

def analyze_co2_variants(input_csv, output_csv, start_mass):

    df = pd.read_csv(input_csv)

    # Required columns
    required_cols = {"Feature_X", "Feature_Y", "Formula"}
    if not required_cols.issubset(df.columns):
        print(f"⚠ Skipping {os.path.basename(input_csv)}: missing columns")
        return

    # Clean numeric
    df["Feature_X"] = pd.to_numeric(df["Feature_X"], errors="coerce")
    df["Feature_Y"] = pd.to_numeric(df["Feature_Y"], errors="coerce")
    df = df.dropna(subset=["Feature_X", "Feature_Y"]).copy()

    # Round masses
    df["Feature_X_round"] = df["Feature_X"].round(ROUND_DECIMALS)
    df["Feature_Y_round"] = df["Feature_Y"].round(ROUND_DECIMALS)

    # Filter CO2 edges
    df_co2 = df[df["Formula"].astype(str).apply(is_co2_variant)]

    if df_co2.empty:
        print(f"⚠ No CO2 edges: {os.path.basename(input_csv)}")
        return

    # Start traversal
    start_mass = round(float(start_mass), ROUND_DECIMALS)

    paths = trace_co2_variants(
        df_co2,
        start_mass,
        pathway={},
        step=1,
        visited=set()
    )

    if not paths:
        print(f"⚠ No CO2 paths from {start_mass}")
        return

    # Save
    out_df = pd.DataFrame(paths)
    out_df.to_csv(output_csv, index=False)

    print(f"✔ Saved {len(out_df)} CO2 paths → {os.path.basename(output_csv)}")


# ============================================
# FILENAME MASS EXTRACTION
# ============================================

def extract_mz_from_filename(filename):
    match = re.search(r"_(\d+\.\d+)", filename)
    return float(match.group(1)) if match else None


# ============================================
# RUN PIPELINE
# ============================================

input_folder = r"C:/Users/kusha/OneDrive/Desktop/FTMS data analysis/Open ocean/rearranged"
output_folder = os.path.join(input_folder, "CO2_Variant_Pathways")
os.makedirs(output_folder, exist_ok=True)

print("\n🚀 Extracting ALL CO2 variant pathways...\n")

for file in os.listdir(input_folder):

    if not file.endswith(".csv"):
        continue

    mz_val = extract_mz_from_filename(file)

    if mz_val is None:
        print(f"⚠ Skipping {file} (no mass found)")
        continue

    in_path = os.path.join(input_folder, file)
    out_name = f"CO2_paths_{mz_val:.5f}.csv"
    out_path = os.path.join(output_folder, out_name)

    analyze_co2_variants(in_path, out_path, mz_val)

print("\n✅ DONE\n")